### Importing Dependencies

In [ ]:
import json 
import gzip 
import pandas as pd


### Working with downloaded data

In [ ]:
 with gzip.open("../../data/meta_Electronics.jsonl.gz", 'rt') as f:
     first_line = json.loads(f.readline())
     print(first_line)

### Filter out only recent items only after 2022 (appeared in stock in 2022 or later (2023 for dataset limitations))

In [ ]:
def filter_data_by_date(data: dict) -> bool:
    filter = False
    if int(data["details"]["Date First Available"][-4:]) < 2022:
        filter = True
    return filter


In [ ]:
test_dict = {'main_category': 'All Electronics',
 'title': 'FS-1051 FATSHARK TELEPORTER V3 HEADSET',
 'average_rating': 3.5,
 'rating_number': 6,
 'features': [],
 'description': ['Teleporter V3 The "Teleporter V3" kit sets a new level of value in the FPV world with Fatshark'],
 'price': None,
 'images': [{'thumb': 'https://m.media-amazon.com/images/I/41qrX56lsYL._AC_US40_.jpg',
   'large': 'https://m.media-amazon.com/images/I/41qrX56lsYL._AC_.jpg',
   'variant': 'MAIN',
   'hi_res': None}],
 'videos': [],
 'store': 'Fat Shark',
 'categories': ['Electronics', 'Television & Video', 'Video Glasses'],
 'details': {'Date First Available': 'August 2, 2014',
  'Manufacturer': 'Fatshark'},
 'parent_asin': 'B00MCW7G9M',
 'bought_together': None
 }


filter_data_by_date(test_dict)

In [ ]:
with gzip.open("../../data/meta_Electronics.jsonl.gz", 'rt') as fp:
    with open("../../data/meta_Electronics_2022_2023.jsonl", 'a', encoding='utf-8') as fp_out:
        with open("../../data/meta_Electronics_2022_2023_no_date.jsonl", 'a', encoding='utf-8') as fp_out_no_date:
            i = 0
            for line in fp:
                data = json.loads(line.strip())
                try:
                    filter = filter_data_by_date(data)
                    if not filter:
                        json.dump(data, fp_out)
                        fp_out.write('\n')
                        fp_out.flush()
                except:
                    json.dump(data, fp_out_no_date)
                    fp_out_no_date.write('\n')
                    fp_out_no_date.flush()
                i += 1
                if i % 10000 == 0:
                    print(f"Processed {i} lines")


# Split the items into two categories one has main category and the other does not 

In [ ]:
 def filter_category(data: dict) -> dict:
     filter = False
     if data["main_category"] == None:
         filter = True
     return filter

### filter out items that do not have main category

In [ ]:
with open("../../data/meta_Electronics_2022_2023.jsonl", "r") as fp:
    with open("../../data/meta_Electronics_2022_2023_with_category.jsonl", "a", encoding="utf-8") as fp_out:
        with open("../../data/meta_Electronics_2022_2023_no_category.jsonl", "a", encoding="utf-8") as fp_out_no_category:
            i = 0

            for line in fp:
                data = json.loads(line.strip())

                if not filter_category(data):
                    json.dump(data, fp_out)
                    fp_out.write('\n')
                    fp_out.flush()
                else:
                    json.dump(data, fp_out_no_category)
                    fp_out_no_category.write('\n')
                    fp_out_no_category.flush()

                i += 1

                if i % 10000 == 0:
                    print(f"Processed {i} lines")

#### explore distribution by categories 


In [ ]:
# with open("../../data/meta_Electronics_2022_2023_with_category.jsonl", "r") as fp:
#     first_line = json.loads(fp.readline())
#     print(first_line)
    
df = pd.read_json("../../data/meta_Electronics_2022_2023_with_category.jsonl", lines=True)
print(len(df))

In [ ]:
df.head() #top 5 where each col is equalto the name of the json key

In [ ]:
df["main_category"].value_counts().plot(kind="bar")

### Items with only 100 ratings or more

In [ ]:
df_ratings_100 = df[df["rating_number"] >= 100]

In [ ]:
len(df_ratings_100)

In [ ]:
df_ratings_100["main_category"].value_counts().plot(kind="bar")

#### Explore distribution of ratings

In [ ]:
df_ratings_100["average_rating"].plot(kind="hist", bins=50, range=(0, 5))

#### Sample 1000

In [ ]:
df_sample_1000 = df_ratings_100.sample(n=1000, random_state=42)
print(len(df_sample_1000))

In [ ]:
len(df_sample_100)
len(df_sample_1000) 

In [ ]:
df_sample_1000["average_rating"].plot(kind="hist", bins=50, range=(0, 5))   

In [ ]:
df_sample_1000["price"].plot(kind="hist", bins=100, range=(0, 500))

In [ ]:
print(len(df_sample_1000))
df_ratings_100.to_json("../../data/meta_Electronics_2022_2023_with_category_ratings_100.jsonl", orient="records", lines=True)
df_sample_1000.to_json("../../data/meta_Electronics_2022_2023_with_category_ratings_100_sample_1000.jsonl", orient="records", lines=True)

### filter out the ratings in Electronics.jsonl.gz such that the ratings are just the ones from the 1000 dataset

In [ ]:
#parent_asin
# df_ratings_100 = pd.read_json("../../data/meta_Electronics_2022_2023_with_category_ratings_100.jsonl", orient="records", lines=True)
df_sample_1000 = pd.read_json("../../data/meta_Electronics_2022_2023_with_category_ratings_100_sample_1000.jsonl", orient="records", lines=True)

In [ ]:
with gzip.open("../../data/Electronics.jsonl.gz", 'r') as fp:
    with open("../../data/Electronics_2022_2023_with_category_ratings_100_sample_1000.jsonl", 'a') as fp_out:
        id_list = set(df_sample_1000['parent_asin'].values)

        i = 0
        for line in fp:
            data = json.loads(line.strip())

            if data['parent_asin'] in id_list:
                json.dump(data, fp_out)
                fp_out.write('\n')
                fp_out.flush()

            i += 1

            if i % 100000 == 0:
                print(f"Processed {i} lines")

In [ ]:
test = pd.read_json("../../data/Electronics_2022_2023_with_category_ratings_100_sample_1000.jsonl", lines=True)
print(len(test))